# Lab - Prompt Caching: The Cost Model

Three tasks that turn the lecture's three usage lines into an architecture decision: what a request is made of, what each kind costs, and which cache lifetime fits the traffic.

| Task | What you learn |
| --- | --- |
| 1 | Four requests, one table: no cache, a write, two reads |
| 2 | Price each kind per thousand requests, and the break-even of a write |
| 3 | Replay a busy day and a quiet one: 5-minute or 1-hour cache |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
POLICY = Path("policy.txt").read_text()   # sixteen sections, the same for every question
INSTRUCTION = "You are ShopAssist. Answer from the policy below in one sentence."
QUESTIONS = ["Can I return earrings?",
             "How long does a card refund take?",
             "Who approves a refund of 250?"]

# Prices for this model, dollars per million tokens, and the cache multipliers the
# lecture gives: a write costs a quarter more than normal input, a read a twentieth.
PRICE_IN, PRICE_OUT = 2.00, 10.00
CACHE_WRITE, CACHE_READ = 1.25, 0.05
CACHE_WRITE_1H = 2.00   # a cache that lives an hour costs twice the input price to write

CACHED_SYSTEM = [   # the lecture's two blocks: everything up to the mark is cached
    {"type": "text", "text": INSTRUCTION},
    {"type": "text", "text": POLICY, "cache_control": {"type": "ephemeral"}},
]


def ask(question, system):
    return client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=system,
        messages=[{"role": "user", "content": question}],
    )

---

## Task 1 - What one request is made of

In [ ]:
# ============================================================
# TASK 1 - What one request is made of
# ============================================================
#
# WHAT TO DO
#   Send the first question once without the cache - the
#   policy as plain text - then all three with the cached
#   system blocks. Keep one row per request: normal input,
#   cache write, cache read, output.
#
# WHY IT MATTERS
#   Before you can price a design you need its parts. The
#   uncached request bills the whole policy as input. The first
#   cached one writes it; the next ones read it, and only the
#   question is normal input. These four rows are the numbers
#   the cost model in Task 2 runs on.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Request two, a minute later, with another question: the
#   same prefix".
#
# HOW TO DO IT
#   Three blanks:
#
#       INSTRUCTION + "\n\n" + POLICY      the same text, no mark
#       u.cache_creation_input_tokens      what this request wrote
#       u.cache_read_input_tokens          what it read
# ============================================================

shopassist_lab.reset()   # provided, offline only: start with an empty prompt cache

try:
    # BEGIN SOLUTION
    def row(label, response):
        u = response.usage
        return {"request": label, "input": u.input_tokens,
                "cache_write": u.cache_creation_input_tokens,
                "cache_read": u.cache_read_input_tokens,
                "output": u.output_tokens}

    rows = [row("no cache", ask(QUESTIONS[0], INSTRUCTION + "\n\n" + POLICY))]
    # SCAFFOLD: def row(label, response):
    # SCAFFOLD:     u = response.usage
    # SCAFFOLD:     return {"request": label, "input": u.input_tokens,
    # SCAFFOLD:             "cache_write": ...,   # u.cache_creation_input_tokens
    # SCAFFOLD:             "cache_read": ...,   # u.cache_read_input_tokens
    # SCAFFOLD:             "output": u.output_tokens}
    # SCAFFOLD:
    # SCAFFOLD: rows = [row("no cache", ask(QUESTIONS[0], ...))]   # INSTRUCTION + "\n\n" + POLICY
    # END SOLUTION: replace each ... below with the value named beside it
    for n, question in enumerate(QUESTIONS, 1):
        rows.append(row(f"cached {n}", ask(question, CACHED_SYSTEM)))
    if any(... in r.values() for r in rows):   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(f"{'request':10} {'input':>6} {'write':>6} {'read':>6} {'output':>6}")
for r in rows:
    print(f"{r['request']:10} {r['input']:6} {r['cache_write']:6} {r['cache_read']:6} {r['output']:6}")

check("cache_rows", rows=rows)

---

## Task 2 - Price every kind of request

In [ ]:
# ============================================================
# TASK 2 - Price every kind of request
# ============================================================
#
# WHAT TO DO
#   Write cost(): normal input at the input price, a cache
#   write at 1.25 times it, a cache read at a twentieth, output
#   at the output price. Price the three kinds of request, then
#   work out how many reads pay back the extra cost of a write.
#
# WHY IT MATTERS
#   The write is the investment: it costs more than an uncached
#   request. Every read returns most of the policy's price. The
#   break-even tells you whether caching pays for a given
#   traffic - and the per-thousand price is the number a
#   stakeholder can compare.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The policy was read from the cache at a small fraction of
#   the price".
#
# HOW TO DO IT
#   Three blanks:
#
#       CACHE_WRITE                         the write multiplier
#       CACHE_READ                          the read multiplier
#       premium / saving_per_read           reads that repay one write
# ============================================================

try:
    # BEGIN SOLUTION
    def cost(r):
        dollars = (r["input"] * PRICE_IN
                   + r["cache_write"] * PRICE_IN * CACHE_WRITE
                   + r["cache_read"] * PRICE_IN * CACHE_READ
                   + r["output"] * PRICE_OUT)
        return dollars / 1_000_000

    policy_tokens = rows[1]["cache_write"]
    premium = policy_tokens * PRICE_IN * (CACHE_WRITE - 1)
    saving_per_read = policy_tokens * PRICE_IN * (1 - CACHE_READ)
    break_even = premium / saving_per_read
    # SCAFFOLD: def cost(r):
    # SCAFFOLD:     dollars = (r["input"] * PRICE_IN
    # SCAFFOLD:                + r["cache_write"] * PRICE_IN * ...   # CACHE_WRITE
    # SCAFFOLD:                + r["cache_read"] * PRICE_IN * ...   # CACHE_READ
    # SCAFFOLD:                + r["output"] * PRICE_OUT)
    # SCAFFOLD:     return dollars / 1_000_000
    # SCAFFOLD:
    # SCAFFOLD: policy_tokens = rows[1]["cache_write"]
    # SCAFFOLD: premium = policy_tokens * PRICE_IN * (CACHE_WRITE - 1)
    # SCAFFOLD: saving_per_read = policy_tokens * PRICE_IN * (1 - CACHE_READ)
    # SCAFFOLD: break_even = ...   # premium / saving_per_read
    # END SOLUTION: replace each ... below with the value named beside it

    per_1k = {"no cache": cost(rows[0]) * 1000, "cache write": cost(rows[1]) * 1000,
              "cache read": cost(rows[2]) * 1000}
    if break_even is ...:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
for kind, dollars in per_1k.items():
    print(f"{kind:12} ${dollars:6.2f} per 1,000 requests")
print(f"one write is repaid after {break_even:.2f} reads")

check("cache_prices", per_1k=per_1k, break_even=break_even, cost=cost)

---

## Task 3 - Choose the cache lifetime by the traffic

In [ ]:
# ============================================================
# TASK 3 - Choose the cache lifetime by the traffic
# ============================================================
#
# WHAT TO DO
#   Replay a day of requests against a cache that lives 5
#   minutes, and one that lives an hour; every hit refreshes the
#   lifetime. Count the writes and reads, price the day, and pick
#   the cheaper lifetime for a busy queue and a quiet one.
#
# WHY IT MATTERS
#   The lecture: the cache lives five minutes after its last
#   use. A busy assistant keeps it warm all day. A quiet one -
#   a request every twenty minutes - pays the write on every
#   call, which costs more than no cache at all. An hour-long
#   cache costs more to write, and fits that traffic.
#
# WHERE TO SEE IT IN THE LECTURE
#   "A quiet one pays the write again after a pause".
#
# HOW TO DO IT
#   Three blanks:
#
#       minute - last <= ttl    still alive: a read that refreshes it
#       CACHE_WRITE_1H          what an hour-long write costs
#       min(options, key=options.get)    the cheaper lifetime
# ============================================================

TRAFFIC = {   # provided: minutes of the day at which requests arrive, over 8 hours
    "busy queue": list(range(0, 480, 2)),     # every 2 minutes
    "quiet queue": list(range(0, 480, 20)),   # every 20 minutes
}
try:
    QUESTION_IN, OUTPUT = rows[2]["input"], rows[2]["output"]   # provided: a typical question
    # BEGIN SOLUTION
    def replay(minutes, ttl):
        writes = reads = 0
        last = None
        for minute in minutes:
            if last is not None and minute - last <= ttl:
                reads += 1
            else:
                writes += 1
            last = minute
        return writes, reads

    def day_cost(minutes, ttl, write_multiplier):
        writes, reads = replay(minutes, ttl)
        tokens = (writes * policy_tokens * write_multiplier + reads * policy_tokens * CACHE_READ
                  + len(minutes) * QUESTION_IN) * PRICE_IN + len(minutes) * OUTPUT * PRICE_OUT
        return round(tokens / 1_000_000, 4), writes, reads

    plan = {}
    for shape, minutes in TRAFFIC.items():
        options = {"5 minutes": day_cost(minutes, 5, CACHE_WRITE)[0],
                   "1 hour": day_cost(minutes, 60, CACHE_WRITE_1H)[0]}
        plan[shape] = {"options": options, "choose": min(options, key=options.get),
                       "writes_5m": replay(minutes, 5)[0]}
    # SCAFFOLD: def replay(minutes, ttl):
    # SCAFFOLD:     writes = reads = 0
    # SCAFFOLD:     last = None
    # SCAFFOLD:     for minute in minutes:
    # SCAFFOLD:         if last is not None and ...:   # minute - last <= ttl
    # SCAFFOLD:             reads += 1
    # SCAFFOLD:         else:
    # SCAFFOLD:             writes += 1
    # SCAFFOLD:         last = minute
    # SCAFFOLD:     return writes, reads
    # SCAFFOLD:
    # SCAFFOLD: def day_cost(minutes, ttl, write_multiplier):
    # SCAFFOLD:     writes, reads = replay(minutes, ttl)
    # SCAFFOLD:     tokens = (writes * policy_tokens * write_multiplier + reads * policy_tokens * CACHE_READ
    # SCAFFOLD:               + len(minutes) * QUESTION_IN) * PRICE_IN + len(minutes) * OUTPUT * PRICE_OUT
    # SCAFFOLD:     return round(tokens / 1_000_000, 4), writes, reads
    # SCAFFOLD:
    # SCAFFOLD: plan = {}
    # SCAFFOLD: for shape, minutes in TRAFFIC.items():
    # SCAFFOLD:     options = {"5 minutes": day_cost(minutes, 5, CACHE_WRITE)[0],
    # SCAFFOLD:                "1 hour": day_cost(minutes, 60, ...)[0]}   # CACHE_WRITE_1H
    # SCAFFOLD:     plan[shape] = {"options": options, "choose": ...,   # min(options, key=options.get)
    # SCAFFOLD:                    "writes_5m": replay(minutes, 5)[0]}
    # END SOLUTION: replace each ... below with the value named beside it

    no_cache = {shape: round(len(m) * cost(rows[0]), 4) for shape, m in TRAFFIC.items()}
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for shape, p in plan.items():
    print(f"{shape:11} no cache ${no_cache[shape]:.4f} | 5 minutes ${p['options']['5 minutes']:.4f}"
          f" | 1 hour ${p['options']['1 hour']:.4f} -> {p['choose']}")

check("cache_ttl", plan=plan, no_cache=no_cache, traffic=TRAFFIC)

---

## Done

Measure what a request is made of, price each kind, and choose the cache lifetime by the traffic: a busy assistant keeps a 5-minute cache warm; a quiet one needs the hour-long cache, or no cache at all.